# Download AutoPET-I (FDG-PET-CT-Lesions) from FDAT to Google Drive

FDAT redirects to a signed S3 URL (expires 60s). This notebook gets a fresh
signed URL per 1 GB part, downloads with `curl` to Colab `/tmp`, uploads to
Drive via API, then deletes from `/tmp`. Fully resumable across sessions.

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN

In [ ]:
# Step 1: Authenticate
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import os, time, subprocess, requests

drive_service = build('drive', 'v3')
print('Authenticated')

In [ ]:
# Step 2: Create/find Drive folders
def get_or_create_folder(name, parent_id=None):
    query = f"name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    if parent_id:
        query += f" and '{parent_id}' in parents"
    results = drive_service.files().list(q=query, fields='files(id)').execute()
    if results.get('files'):
        return results['files'][0]['id']
    metadata = {'name': name, 'mimeType': 'application/vnd.google-apps.folder'}
    if parent_id:
        metadata['parents'] = [parent_id]
    return drive_service.files().create(body=metadata, fields='id').execute()['id']

data_folder_id = get_or_create_folder(WORK_DIR_FOLDER_NAME)
dest_id = get_or_create_folder('autopet_i', data_folder_id)
parts_id = get_or_create_folder('_zip_parts', dest_id)
print(f'Parts folder ID: {parts_id}')

In [ ]:
# Step 3: Config and resume check

FDAT_URL = 'https://fdat.uni-tuebingen.de/records/wf9fy-txq84/files/fdg-pet-ct-lesions.zip?download=1'
TOTAL_SIZE = 303_778_740_316
PART_SIZE = 1 * 1024 * 1024 * 1024
TOTAL_PARTS = (TOTAL_SIZE + PART_SIZE - 1) // PART_SIZE

existing_parts = set()
page_token = None
while True:
    results = drive_service.files().list(
        q=f"'{parts_id}' in parents and trashed=false",
        fields='nextPageToken, files(name)',
        pageToken=page_token,
        pageSize=1000
    ).execute()
    for f in results.get('files', []):
        if f['name'].startswith('part_') and f['name'].endswith('.bin'):
            existing_parts.add(f['name'])
    page_token = results.get('nextPageToken')
    if not page_token:
        break

print(f'Total: {TOTAL_SIZE/(1024**3):.1f} GB | Parts: {TOTAL_PARTS}')
print(f'Already uploaded: {len(existing_parts)}')
print(f'Remaining: {TOTAL_PARTS - len(existing_parts)}')

In [ ]:
# Step 4: Download and upload parts
#
# For each part:
#   1. GET fresh signed S3 URL from FDAT (follows 302 redirect)
#   2. curl S3 URL with Range header -> /tmp (1 GB, ~20-30 MB/s)
#   3. Upload /tmp file to Drive via API (~60 MB/s)
#   4. Delete /tmp file

def get_signed_s3_url():
    """Follow FDAT redirect to get a fresh signed S3 URL (expires in 60s)."""
    r = requests.head(FDAT_URL, allow_redirects=True, timeout=30)
    r.raise_for_status()
    return r.url

start_time = time.time()
uploaded_this_session = 0
failed_parts = []

for part_num in range(TOTAL_PARTS):
    part_name = f'part_{part_num:04d}.bin'

    if part_name in existing_parts:
        continue

    start_byte = part_num * PART_SIZE
    end_byte = min(start_byte + PART_SIZE, TOTAL_SIZE) - 1
    part_bytes = end_byte - start_byte + 1
    tmp_path = f'/tmp/{part_name}'

    print(f'\nPart {part_num}/{TOTAL_PARTS} | '
          f'{start_byte/(1024**3):.1f}-{(end_byte+1)/(1024**3):.1f} GB')

    # Get fresh signed URL
    try:
        s3_url = get_signed_s3_url()
    except Exception as e:
        print(f'  Failed to get signed URL: {e}')
        failed_parts.append(part_num)
        time.sleep(10)
        continue

    # Download with curl using Range header on S3 URL
    print(f'  Downloading...')
    dl_start = time.time()

    result = subprocess.run([
        'curl', '-s', '-o', tmp_path,
        '-H', f'Range: bytes={start_byte}-{end_byte}',
        '--retry', '3',
        '--retry-delay', '5',
        '--connect-timeout', '30',
        '--max-time', '600',
        s3_url
    ], capture_output=True, text=True, timeout=660)

    if result.returncode != 0 or not os.path.exists(tmp_path):
        print(f'  curl FAILED (exit {result.returncode}): {result.stderr[:200]}')
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        failed_parts.append(part_num)
        continue

    actual_size = os.path.getsize(tmp_path)
    if actual_size < part_bytes * 0.99:
        print(f'  Incomplete: got {actual_size/(1024**2):.0f} MB, '
              f'expected {part_bytes/(1024**2):.0f} MB')
        os.remove(tmp_path)
        failed_parts.append(part_num)
        continue

    dl_time = time.time() - dl_start
    dl_speed = actual_size / dl_time / (1024**2)
    print(f'  Downloaded: {actual_size/(1024**2):.0f} MB in '
          f'{dl_time:.0f}s ({dl_speed:.1f} MB/s)')

    # Upload to Drive
    print(f'  Uploading to Drive...')
    up_start = time.time()

    media = MediaFileUpload(
        tmp_path, mimetype='application/octet-stream', resumable=True
    )
    file_metadata = {'name': part_name, 'parents': [parts_id]}
    req = drive_service.files().create(
        body=file_metadata, media_body=media, fields='id'
    )

    response = None
    while response is None:
        try:
            status, response = req.next_chunk()
        except Exception as e:
            print(f'    Upload error: {e}, retrying in 5s...')
            time.sleep(5)

    up_time = time.time() - up_start
    up_speed = actual_size / up_time / (1024**2)

    os.remove(tmp_path)
    uploaded_this_session += 1
    existing_parts.add(part_name)

    total_done = len(existing_parts)
    pct = total_done / TOTAL_PARTS * 100
    elapsed = time.time() - start_time
    parts_per_hour = uploaded_this_session / (elapsed / 3600)
    remaining = TOTAL_PARTS - total_done
    eta_hours = remaining / parts_per_hour if parts_per_hour > 0 else 0

    print(f'  DONE | dl:{dl_speed:.0f} up:{up_speed:.0f} MB/s | '
          f'{total_done}/{TOTAL_PARTS} ({pct:.0f}%) | '
          f'~{eta_hours:.1f}h remaining')

elapsed = time.time() - start_time
print(f'\n{"="*60}')
print(f'Session: {uploaded_this_session} parts in {elapsed/3600:.1f}h')
print(f'Total: {len(existing_parts)}/{TOTAL_PARTS}')
if failed_parts:
    print(f'Failed: {failed_parts} (will retry on next run)')
if len(existing_parts) >= TOTAL_PARTS:
    print('\nALL PARTS UPLOADED — proceed to Step 5.')
else:
    print(f'{TOTAL_PARTS - len(existing_parts)} remaining. Re-run to continue.')

In [ ]:
# Step 5: Reassemble parts into original zip on Google Drive
# Run ONLY after all parts uploaded.

print('Listing all parts...')
all_parts = []
page_token = None
while True:
    results = drive_service.files().list(
        q=f"'{parts_id}' in parents and trashed=false and name contains 'part_'",
        fields='nextPageToken, files(id, name, size)',
        pageToken=page_token,
        orderBy='name',
        pageSize=1000
    ).execute()
    all_parts.extend(results.get('files', []))
    page_token = results.get('nextPageToken')
    if not page_token:
        break

all_parts.sort(key=lambda x: x['name'])
total_size = sum(int(p['size']) for p in all_parts)
print(f'Found {len(all_parts)} parts, total {total_size/(1024**3):.1f} GB')

if len(all_parts) < TOTAL_PARTS:
    print(f'Expected {TOTAL_PARTS} parts — upload more first.')
else:
    file_metadata = {'name': 'fdg-pet-ct-lesions.zip', 'parents': [dest_id]}
    headers_init = {
        'Authorization': f'Bearer {drive_service._http.credentials.token}',
        'Content-Type': 'application/json; charset=UTF-8',
        'X-Upload-Content-Type': 'application/zip',
        'X-Upload-Content-Length': str(total_size),
    }
    init_resp = requests.post(
        'https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable',
        headers=headers_init,
        json=file_metadata
    )
    init_resp.raise_for_status()
    upload_url = init_resp.headers['Location']
    print('Resumable upload session created')

    uploaded = 0
    for i, part in enumerate(all_parts):
        dl_req = drive_service.files().get_media(fileId=part['id'])
        tmp_path = '/tmp/current_part.bin'
        with open(tmp_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, dl_req)
            done = False
            while not done:
                _, done = downloader.next_chunk()

        part_size = os.path.getsize(tmp_path)
        with open(tmp_path, 'rb') as fh:
            part_data = fh.read()

        end = uploaded + part_size - 1
        content_range = f'bytes {uploaded}-{end}/{total_size}'
        up_resp = requests.put(
            upload_url,
            headers={'Content-Range': content_range, 'Content-Type': 'application/zip'},
            data=part_data,
            timeout=300
        )

        uploaded += part_size
        os.remove(tmp_path)
        if (i + 1) % 10 == 0 or i == len(all_parts) - 1:
            print(f'  {i+1}/{len(all_parts)} | {uploaded/(1024**3):.1f} GB')

    print(f'\nReassembly complete: {uploaded/(1024**3):.1f} GB')

In [ ]:
# Step 6: Verify
verify = drive_service.files().list(
    q=f"name='fdg-pet-ct-lesions.zip' and '{dest_id}' in parents and trashed=false",
    fields='files(id, name, size, md5Checksum)'
).execute()

if verify.get('files'):
    f = verify['files'][0]
    print(f'Size: {int(f["size"])/(1024**3):.1f} GB')
    md5 = f.get('md5Checksum', 'pending')
    print(f'MD5: {md5}')
    print(f'Expected: eb97d6d6d697f17a09c8f2f4a332f2a0')
    print('VERIFIED' if md5 == 'eb97d6d6d697f17a09c8f2f4a332f2a0' else 'Check again shortly')
else:
    print('File not found — run Step 5')